# 40 - Pre-Train ResNet18 pada FER2013

**Tujuan:** Pre-train ResNet18 di FER2013 (35K gambar emosi) sebelum fine-tune ke dataset sendiri.
Ini memberikan fitur yang lebih relevan (ekspresi wajah) dibanding ImageNet (kucing, mobil, dll).

**Pipeline:** ImageNet -> FER2013 -> Dataset sendiri (two-stage TL)

In [ ]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from training.models import EmotionCNNTransfer
from training.utils import train_model, full_evaluation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

FER_DIR = PROJECT_ROOT / "data" / "benchmark" / "fer2013_prepared"
OUTPUT_DIR = PROJECT_ROOT / "models" / "pretrained"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 10
LR = 0.0001
NUM_CLASSES = 7
EMOTIONS = ["neutral", "happy", "sad", "angry", "fearful", "disgusted", "surprised"]
print("Setup complete.")

## Load FER2013

In [ ]:
# Load FER2013
train_img = np.load(FER_DIR / "X_train_images.npy")
train_y = np.load(FER_DIR / "y_train.npy")
val_img = np.load(FER_DIR / "X_val_images.npy")
val_y = np.load(FER_DIR / "y_val.npy")

print(f"Train: {len(train_y)} | Val: {len(val_y)}")
print(f"Distribution: {dict(sorted(Counter(train_y.tolist()).items()))}")

# DataLoaders
def make_loader(images, labels, batch_size=32, shuffle=True):
    img_t = torch.from_numpy(images).permute(0, 3, 1, 2)
    y_t = torch.from_numpy(labels).long()
    return DataLoader(TensorDataset(img_t, y_t), batch_size=batch_size,
                      shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader = make_loader(train_img, train_y, BATCH_SIZE)
val_loader = make_loader(val_img, val_y, BATCH_SIZE, False)
print("Data loaded.")

## Pre-Train ResNet18 on FER2013

In [ ]:
# Pre-train ResNet18 (ImageNet -> FER2013)
model = EmotionCNNTransfer(num_classes=NUM_CLASSES, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5, min_lr=1e-7)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Pre-training on FER2013...")

history, best_epoch = train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, model_type="cnn", epochs=EPOCHS, patience=PATIENCE,
    save_path=str(OUTPUT_DIR / "resnet18_fer2013.pth"))

print(f"\nBest epoch: {best_epoch}")
print(f"Saved: {OUTPUT_DIR / 'resnet18_fer2013.pth'}")

## Evaluate on FER2013 Test Set

In [ ]:
# Evaluate
test_img = np.load(FER_DIR / "X_test_images.npy") if (FER_DIR / "X_test_images.npy").exists() else val_img
test_y = np.load(FER_DIR / "y_test.npy") if (FER_DIR / "y_test.npy").exists() else val_y
test_loader = make_loader(test_img, test_y, BATCH_SIZE, False)

model.load_state_dict(torch.load(OUTPUT_DIR / "resnet18_fer2013.pth", map_location=device, weights_only=True))
results = full_evaluation(model, test_loader, criterion, device, "cnn", EMOTIONS)

print(f"\nFER2013 Test Results:")
print(f"  Accuracy:  {results['test_accuracy']:.4f}")
print(f"  Macro F1:  {results['test_macro_f1']:.4f}")
print(f"  W-F1:      {results['test_weighted_f1']:.4f}")
print(f"\nPre-trained weights ready at: {OUTPUT_DIR / 'resnet18_fer2013.pth'}")